# Vaani Track 1 - v2 on Kaggle

ATST-Frame + BEATs fusion -> trident span-regression head -> count-head selection ->
per-district calibration.

## Before you run anything

1. **Data, once:** run [`Vaani_Data_Kaggle.ipynb`](Vaani_Data_Kaggle.ipynb) and *Save
   Version*. It downloads the corpus, labels speech (VAD), builds the synthetic clips
   and packs everything into a few files, because Kaggle keeps at most 500 output
   files. Then in *this* notebook: *Add Data -> Your Work -> vaani-data*. The cell
   below finds it on its own.
2. **Accelerator:** *Settings -> Accelerator -> GPU T4 x2*.
3. **Internet:** *Settings -> Internet -> On* (clone + encoder checkpoints).
4. **Everything you edit lives in the CONFIG cell.**

## Where the time goes, and why it fits in a session now

Both T4s train (DDP, one process per GPU, `BATCH_SIZE` per GPU) *and* both validate:
each scores half the held-out set. Validation scores the EMA weights only, every 5th
epoch and the last (`eval_every` in the config). The encoders
run fused attention, the unfrozen phase no longer recomputes the frozen encoder
stack, and no session spends its first hour re-downloading the corpus. The training
log prints throughput, data-wait and GPU memory every 100 steps.

If a session still runs out, `state.pt` is written every epoch and `RESUME_FROM`
continues losslessly - optimiser, EMA and the step counter included.


In [ ]:
import time
SESSION_T0 = time.time()          # the time guard below counts from here
# ============================== CONFIG ==============================
REPO   = "raut7218/vaani-sed-v2"
BRANCH = "speed"

# The packed corpus from Vaani_Data_Kaggle.ipynb. Empty = auto-detect under /kaggle/input.
DATA_FROM = ""                # e.g. "/kaggle/input/vaani-data/vaani"
USE_SYNTHETIC = True          # train on the packed synthetic clips too

# --- training -----------------------------------------------------------
FOLD        = 0               # state-grouped k-fold; train several and ensemble
EPOCHS      = 50              # ~11 h on 2x T4 at BATCH_SIZE 48
BATCH_SIZE  = 48              # PER GPU: 48 x 2 = 96 per optimiser step
RESUME_FROM = ""              # e.g. "/kaggle/input/<this-notebook>/runs/f0"
RUN_TESTS   = False           # the component + overfit suites (~3 min)
SESSION_HOURS = 11.5          # Kaggle stops at 12 h and a killed commit keeps no
                              # output: training ends after the last epoch that
                              # fits, with state.pt written for RESUME_FROM

# --- inference ----------------------------------------------------------
TEST_AUDIO_DIR = ""           # see "find the evaluation audio" below
ENSEMBLE_CKPTS = []           # e.g. ["/kaggle/input/vaani-run-f1/runs/f1/best.pt"]
# =====================================================================

import os, shutil, subprocess
from pathlib import Path

WORK = "/kaggle/working"
if not DATA_FROM:
    hits = sorted(p.parent for p in Path("/kaggle/input").glob("**/manifest.jsonl")
                  if p.parent.name != "synth" and "vad.f16" in os.listdir(p.parent))
    if not hits:
        raise RuntimeError("No packed corpus under /kaggle/input. Run Vaani_Data_Kaggle.ipynb "
                           "once, then Add Data -> Your Work -> vaani-data here.")
    DATA_FROM = str(hits[0])
DATA  = DATA_FROM
SYNTH = DATA + "/synth"
RUN   = WORK + "/runs/f%d" % FOLD
print("data:", DATA, "| run:", RUN)


## 1. Code and encoders

The clone lives in `/tmp`, not `/kaggle/working`: everything under the working
directory counts against the 500-file output cap, and the only output worth keeping
is the run directory.

ATST-Frame (40 ms tokens) is the primary encoder; BEATs (160 ms) rides along as a
semantic channel. The loader **refuses to run** if under 90% of the ATST checkpoint
lands - expect `loaded 138/138 tensors (100.0%)` in the training log.


In [ ]:
SRC = "/tmp/v2"
shutil.rmtree(SRC, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                "https://github.com/%s.git" % REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip -q install -r requirements.txt 2>&1 | tail -2
!python scripts/fetch_encoders.py --all 2>&1 | tail -4
if RUN_TESTS:
    !python tests/test_components.py 2>&1 | tail -3
    !python tests/test_overfit.py 2>&1 | tail -2


In [ ]:
import yaml, json, collections
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["model"]["beats_dir"] = SRC + "/checkpoints"
cfg["train"]["num_workers"] = 2          # 4 vCPUs shared by the two ranks
Path("configs/kaggle.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump({"train": cfg["train"]}, sort_keys=False))

recs = [json.loads(l) for l in open(DATA + "/manifest.jsonl") if l.strip()]
print(len(recs), "clips |", collections.Counter(r["tier"] for r in recs),
      "| with VAD:", sum("vad_off" in r for r in recs))


## 2. Train

`--batch-size` is **per GPU**. With `RESUME_FROM` set, the previous session's
`state.pt` is copied in and `--resume auto` continues from it.


In [ ]:
extra = ("--extra-data " + SYNTH) if USE_SYNTHETIC and Path(SYNTH).exists() else ""
resume = ""
if RESUME_FROM:
    Path(RUN).mkdir(parents=True, exist_ok=True)
    for f in ("state.pt", "best.pt", "history.json"):
        s = Path(RESUME_FROM) / f
        if s.exists():
            shutil.copy(s, Path(RUN) / f)
            print("restored", f)
    resume = "--resume auto"

# 15 min stay in reserve for diagnostics and Kaggle's own output save.
limit_h = SESSION_HOURS - (time.time() - SESSION_T0) / 3600 - 0.25
print("training time budget: %.2f h" % limit_h)
get_ipython().system(
    "torchrun --standalone --nproc_per_node=2 -m src.train.train "
    "--config configs/kaggle.yaml --data %s %s --out %s --fold %d "
    "--epochs %d --batch-size %d --time-limit-h %.2f %s 2>&1 "
    "| grep --line-buffered -v -E 'Warning|warnings.warn|WeightNorm.apply'"
    % (DATA, extra, RUN, FOLD, EPOCHS, BATCH_SIZE, limit_h, resume))
if not Path(RUN, "best.pt").exists():
    raise RuntimeError("training wrote no best.pt - the first traceback above is the "
                       "real error (every rank prints its own copy).")


### If the session is about to time out

Nothing to do by hand: training stops on its own after the last epoch that fits in
`SESSION_HOURS` and writes `state.pt`. If the log ends with `[time] ... stopping at
epoch N`, add this notebook's output as an input to the next session
(*Add Data -> Your Work -> this notebook*), set
`RESUME_FROM = "/kaggle/input/<this-notebook>/runs/f0"` and run again.


## 3. Diagnose

Constant-baseline comparison, strict vs loose recall (found the event vs placed the
boundaries), boundary error distribution, operating point against the corpus prior,
and the selection oracle.


In [ ]:
!python scripts/diagnose.py --ckpt {RUN}/best.pt --data {DATA} --fold {FOLD} --num-workers 4 --batch-size 64 2>&1 | grep -v -E 'Warning|warnings.warn|WeightNorm.apply'


## 4. Find the evaluation audio

Run this, find your eval set in the listing, and paste the path into `TEST_AUDIO_DIR`.


In [ ]:
for p in sorted(Path("/kaggle/input").glob("*")):
    audio = list(p.rglob("*.wav")) + list(p.rglob("*.flac"))
    print("%7d audio files   %s" % (len(audio), p))
    for sub in sorted({q.parent for q in audio[:400]}):
        print("                    ->", sub)


## 5. Submit

Several checkpoints are fused with **1D weighted box fusion**, never by averaging
posteriors. Per-district calibration is transductive: it groups test clips by the
`State_District` in their filenames and uses only unlabelled test audio.

**Watch the last line.** If `events/clip` and `coverage` are far from the priors (1.22
and 0.52), the operating point is wrong.


In [ ]:
ckpts = [c for c in (ENSEMBLE_CKPTS or [RUN + "/best.pt"]) if Path(c).exists()]
if not TEST_AUDIO_DIR:
    print("SKIPPED: set TEST_AUDIO_DIR in the CONFIG cell and re-run this cell.")
elif not ckpts:
    print("SKIPPED: no checkpoint - train first, or list finished runs in ENSEMBLE_CKPTS.")
else:
    get_ipython().system(
        "python -m src.infer.predict --ckpt %s --audio-dir %s --out %s/submission.zip "
        "--batch-size 32 --num-workers 4" % (" ".join(ckpts), TEST_AUDIO_DIR, WORK))


In [ ]:
import zipfile, numpy as np
if not Path(WORK + "/submission.zip").exists():
    print("no submission.zip - the cell above was skipped.")
else:
    with zipfile.ZipFile(WORK + "/submission.zip") as z:
        assert z.namelist() == ["predictions.jsonl"], z.namelist()
        rows = [json.loads(l) for l in
                z.read("predictions.jsonl").decode().splitlines() if l.strip()]
    ids = [r["clip_id"] for r in rows]
    assert len(ids) == len(set(ids)), "duplicate clip_id"
    for r in rows:
        for e in r["events"]:
            assert e["onset"] >= 0 and e["offset"] >= e["onset"], r["clip_id"]
    print("%d clips | %d events | %.2f events/clip | %.1f%% empty"
          % (len(rows), sum(len(r["events"]) for r in rows),
             np.mean([len(r["events"]) for r in rows]),
             100 * np.mean([not r["events"] for r in rows])))
    print("Download /kaggle/working/submission.zip from the Output tab -> Codabench.")


## 6. More folds for the ensemble

Each fold holds out a different group of states. Train them in separate sessions
(`FOLD = 1`, `2`, ...), then list all their `best.pt` paths in `ENSEMBLE_CKPTS`.
